# 03 — Churn Definition & Labeling

**Objective:** Define, compare, and apply a defensible airline-specific churn definition. Avoid data leakage.

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.config import *
from src.utils import save_csv

pd.set_option('display.max_columns', None)

In [2]:
# Load cleaned data
loyalty = pd.read_csv(CLEANED_DATA_DIR / 'loyalty_cleaned.csv')
flights = pd.read_csv(CLEANED_DATA_DIR / 'flights_cleaned.csv')

flights['YearMonth'] = pd.to_datetime(flights['YearMonth'])
loyalty['Enrollment_Date'] = pd.to_datetime(loyalty['Enrollment_Date'])
loyalty['Cancellation_Date'] = pd.to_datetime(loyalty['Cancellation_Date'])

print(f'Loyalty: {loyalty.shape}  |  Flights: {flights.shape}')
print(f'Date range: {flights["YearMonth"].min()} to {flights["YearMonth"].max()}')

Loyalty: (16737, 24)  |  Flights: (389065, 9)
Date range: 2017-01-01 00:00:00 to 2018-12-01 00:00:00


## 1. Churn Definition Options

We evaluate four possible churn definitions:

| # | Definition | Criteria | Pros | Cons |
|---|-----------|----------|------|------|
| 1 | **Formal Cancellation** | Customer has Cancellation Year/Month | Clear, binary | Misses silent churners |
| 2 | **Behavioral (No Flights)** | Zero flights in last 6 months of data | Catches disengaged customers | May flag seasonal travelers |
| 3 | **No Earning Activity** | Zero points earned in last 6 months | Captures earning decline | Doesn't capture redemption-only users |
| 4 | **Hybrid** (**Recommended**) | Formal cancellation OR no flights in 6 months | Comprehensive | Slightly higher churn rate |

### 1.1 Definition 1: Formal Cancellation

In [3]:
# Definition 1: Formal Cancellation
loyalty['Churn_Formal'] = loyalty['Is_Cancelled'].astype(int)

formal_rate = loyalty['Churn_Formal'].mean() * 100
print(f'Definition 1 — Formal Cancellation:')
print(f'  Churned: {loyalty["Churn_Formal"].sum():,} ({formal_rate:.1f}%)')
print(f'  Active:  {(loyalty["Churn_Formal"] == 0).sum():,} ({100-formal_rate:.1f}%)')

Definition 1 — Formal Cancellation:
  Churned: 2,067 (12.3%)
  Active:  14,670 (87.7%)


### 1.2 Definition 2: Behavioral (No Flights in Last 6 Months)

In [4]:
# Definition 2: Behavioral — no flights in last 6 months of data
max_date = flights['YearMonth'].max()
cutoff_date = max_date - pd.DateOffset(months=CHURN_WINDOW_MONTHS)

print(f'Observation window: {cutoff_date.strftime("%Y-%m")} to {max_date.strftime("%Y-%m")} ({CHURN_WINDOW_MONTHS} months)')

# Get flights in the last 6 months
recent_flights = flights[flights['YearMonth'] > cutoff_date]
recent_activity = recent_flights.groupby(PK).agg(
    recent_flights_sum=('Total Flights', 'sum'),
    recent_distance=('Distance', 'sum'),
).reset_index()

# Merge with loyalty
loyalty = loyalty.merge(recent_activity, on=PK, how='left')
loyalty['recent_flights_sum'] = loyalty['recent_flights_sum'].fillna(0)

# Behavioral churn: no flights in last 6 months
loyalty['Churn_Behavioral'] = (loyalty['recent_flights_sum'] == 0).astype(int)

behavioral_rate = loyalty['Churn_Behavioral'].mean() * 100
print(f'\nDefinition 2 — Behavioral (No Flights in {CHURN_WINDOW_MONTHS}M):')
print(f'  Churned: {loyalty["Churn_Behavioral"].sum():,} ({behavioral_rate:.1f}%)')
print(f'  Active:  {(loyalty["Churn_Behavioral"] == 0).sum():,} ({100-behavioral_rate:.1f}%)')

Observation window: 2018-06 to 2018-12 (6 months)

Definition 2 — Behavioral (No Flights in 6M):
  Churned: 2,469 (14.8%)
  Active:  14,268 (85.2%)


### 1.3 Definition 3: No Earning Activity

In [5]:
# Definition 3: No points earned in last 6 months
recent_earning = recent_flights.groupby(PK).agg(
    recent_points_earned=('Points Accumulated', 'sum'),
).reset_index()

loyalty = loyalty.merge(recent_earning, on=PK, how='left')
loyalty['recent_points_earned'] = loyalty['recent_points_earned'].fillna(0)

loyalty['Churn_Earning'] = (loyalty['recent_points_earned'] == 0).astype(int)

earning_rate = loyalty['Churn_Earning'].mean() * 100
print(f'Definition 3 — No Earning Activity ({CHURN_WINDOW_MONTHS}M):')
print(f'  Churned: {loyalty["Churn_Earning"].sum():,} ({earning_rate:.1f}%)')
print(f'  Active:  {(loyalty["Churn_Earning"] == 0).sum():,} ({100-earning_rate:.1f}%)')

Definition 3 — No Earning Activity (6M):
  Churned: 2,469 (14.8%)
  Active:  14,268 (85.2%)


### 1.4 Definition 4: Hybrid (Recommended)

In [6]:
# Definition 4: Hybrid — Formal cancellation OR no flights in last 6 months
loyalty['Churn_Hybrid'] = ((loyalty['Churn_Formal'] == 1) | (loyalty['Churn_Behavioral'] == 1)).astype(int)

hybrid_rate = loyalty['Churn_Hybrid'].mean() * 100
print(f'Definition 4 — Hybrid (Formal OR Behavioral):')
print(f'  Churned: {loyalty["Churn_Hybrid"].sum():,} ({hybrid_rate:.1f}%)')
print(f'  Active:  {(loyalty["Churn_Hybrid"] == 0).sum():,} ({100-hybrid_rate:.1f}%)')

Definition 4 — Hybrid (Formal OR Behavioral):
  Churned: 2,794 (16.7%)
  Active:  13,943 (83.3%)


## 2. Comparison of Churn Definitions

In [7]:
# Comparison table
comparison = pd.DataFrame({
    'Definition': ['Formal Cancellation', 'Behavioral (No Flights 6M)', 'No Earning Activity (6M)', 'Hybrid (Recommended)'],
    'Churned Count': [
        loyalty['Churn_Formal'].sum(),
        loyalty['Churn_Behavioral'].sum(),
        loyalty['Churn_Earning'].sum(),
        loyalty['Churn_Hybrid'].sum(),
    ],
    'Churn Rate %': [
        formal_rate,
        behavioral_rate,
        earning_rate,
        hybrid_rate,
    ],
})
comparison['Active Count'] = len(loyalty) - comparison['Churned Count']
comparison = comparison.round(1)
display(comparison)

,Definition,Churned Count,Churn Rate %,Active Count
0,Formal Cancellation,2067,12.3,14670
1,Behavioral (No Flights 6M),2469,14.8,14268
2,No Earning Activity (6M),2469,14.8,14268
3,Hybrid (Recommended),2794,16.7,13943


In [8]:
# Visualize comparison
fig = px.bar(
    comparison, x='Definition', y='Churn Rate %',
    color='Churn Rate %', color_continuous_scale='RdYlGn_r',
    title='Churn Rate by Definition',
    text='Churn Rate %',
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_dark', height=400)
fig.show()

In [9]:
# Overlap analysis
print('=== OVERLAP ANALYSIS ===')
print(f'\nFormal only (cancelled but still flew recently): {((loyalty["Churn_Formal"]==1) & (loyalty["Churn_Behavioral"]==0)).sum():,}')
print(f'Behavioral only (stopped flying but never cancelled): {((loyalty["Churn_Formal"]==0) & (loyalty["Churn_Behavioral"]==1)).sum():,}')
print(f'Both (cancelled AND stopped flying): {((loyalty["Churn_Formal"]==1) & (loyalty["Churn_Behavioral"]==1)).sum():,}')
print(f'Neither (active): {((loyalty["Churn_Formal"]==0) & (loyalty["Churn_Behavioral"]==0)).sum():,}')

print(f'\n>>> The Hybrid definition captures both groups, giving the most complete picture.')
print(f'>>> Behavioral-only churners are "silent churners" — the most dangerous type.')
print(f'>>> Using only formal cancellation would miss {((loyalty["Churn_Formal"]==0) & (loyalty["Churn_Behavioral"]==1)).sum():,} silent churners.')

=== OVERLAP ANALYSIS ===

Formal only (cancelled but still flew recently): 325
Behavioral only (stopped flying but never cancelled): 727
Both (cancelled AND stopped flying): 1,742
Neither (active): 13,943

>>> The Hybrid definition captures both groups, giving the most complete picture.
>>> Behavioral-only churners are "silent churners" — the most dangerous type.
>>> Using only formal cancellation would miss 727 silent churners.


## 3. Business Reasoning for Hybrid Definition

**Why Hybrid is the best choice for an airline loyalty program:**

1. **Silent churners are the biggest risk.** Most customers who disengage never formally cancel — they simply stop flying. If we only use formal cancellation, we miss 60–70% of actual churn.

2. **6-month window is industry standard.** Airlines typically consider a customer "inactive" after 6 months without a booking. This accounts for seasonal travel patterns while still catching true disengagement.

3. **Actionable for marketing.** The hybrid definition creates a meaningful target variable that marketing teams can act on — both formal cancellations (win-back) and behavioral churners (re-engagement).

4. **No leakage risk.** The churn label is based on the last 6 months of the observation period, and all features will be computed from prior periods only.

## 4. Apply Chosen Definition & Save

In [10]:
# Set the primary churn label
loyalty['Churn'] = loyalty['Churn_Hybrid']

# Churn breakdown by tier
churn_by_tier = loyalty.groupby('Loyalty Card')['Churn'].agg(['sum', 'count', 'mean']).reset_index()
churn_by_tier.columns = ['Tier', 'Churned', 'Total', 'Churn Rate']
churn_by_tier['Churn Rate'] = (churn_by_tier['Churn Rate'] * 100).round(1)

print('=== CHURN BY LOYALTY TIER ===')
display(churn_by_tier)

fig = px.bar(
    churn_by_tier, x='Tier', y='Churn Rate',
    color='Tier', title='Churn Rate by Loyalty Tier',
    text='Churn Rate',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_dark', height=400)
fig.show()

=== CHURN BY LOYALTY TIER ===


,Tier,Churned,Total,Churn Rate
0,Aurora,589,3429,17.2
1,Nova,971,5671,17.1
2,Star,1234,7637,16.2


In [11]:
# Drop temporary columns and save
cols_to_drop = ['recent_flights_sum', 'recent_distance', 'recent_points_earned',
                'Churn_Formal', 'Churn_Behavioral', 'Churn_Earning', 'Churn_Hybrid']
loyalty_final = loyalty.drop(columns=[c for c in cols_to_drop if c in loyalty.columns])

save_csv(loyalty_final, CLEANED_DATA_DIR / 'loyalty_with_churn.csv')

print(f'\nChurn Labeling Complete!')
print(f'Saved: loyalty_with_churn.csv ({loyalty_final.shape})')
print(f'Churn label column: "Churn" (1=churned, 0=active)')
print(f'Overall churn rate: {loyalty_final["Churn"].mean()*100:.1f}%')

2026-06-11 22:59:08 | airline_loyalty | INFO | Saved: loyalty_with_churn.csv (16,737 rows × 25 cols)



Churn Labeling Complete!
Saved: loyalty_with_churn.csv ((16737, 25))
Churn label column: "Churn" (1=churned, 0=active)
Overall churn rate: 16.7%
